# 🌳 Árboles (Trees): Conceptos Fundamentales y ADT

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap08/01_Conceptos_Arboles_Teoria.ipynb)

### 🎯 Objetivos de aprendizaje

Al finalizar este sección serás capaz de:

1. Definir un árbol formalmente y distinguir sus componentes: raíz, nodos internos, hojas, aristas, caminos.
2. Calcular la profundidad (`depth`) y la altura (`height`) de cualquier nodo.
3. Implementar el ADT `Tree` como clase abstracta en Python.
4. Explicar por qué `height2` (O(n)) es más eficiente que `height1` (O(n²)).
5. Enunciar la Prop. 8.5: la suma de hijos de todos los nodos es n−1.

---

## 📖 Contenidos

**Libro:** Goodrich, Tamassia & Goldwasser — *Data Structures and Algorithms in Python*, §8.1–§8.2  


| Sección | Tema |
|---------|------|
| §8.1 | Terminología general: raíz, padre, hijo, hoja, profundidad, altura |
| §8.2 | ADT de árbol — clase abstracta `Tree` en Python |
| §8.2.1 | `depth(p)` — algoritmo y complejidad O(dₚ + 1) |
| §8.2.2 | `height1` O(n²) vs `height2` O(n) — análisis comparativo |
| — | Proposición 8.5 — verificación numérica |
| — | Tests con `unittest` |

## 🔧 Configuración del entorno

In [ ]:
import sys
from abc import ABCMeta, abstractmethod
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"Python {sys.version}")
print("Librerías cargadas correctamente ✓")

---

## 🌿 §8.1 Terminología fundamental

Un **árbol** T es un conjunto de nodos que almacenan elementos con una relación *padre-hijo*:

- Si T no es vacío, tiene un nodo especial sin padre: la **raíz**.
- Cada nodo v ≠ raíz tiene exactamente **un** padre w.

### Vocabulario esencial

| Término | Definición |
|---------|-----------|
| **Raíz** (root) | Nodo sin padre |
| **Hijo** (child) | v es hijo de u si u es padre de v |
| **Hermanos** (siblings) | Nodos con el mismo padre |
| **Nodo interno** | Tiene al menos un hijo |
| **Hoja** (leaf) | No tiene hijos |
| **Ancestro** | u es ancestro de v si u=v ó u es ancestro del padre de v |
| **Descendiente** | v es descendiente de u si u es ancestro de v |
| **Arista** (edge) | Par (u,v) donde uno es padre del otro |
| **Camino** (path) | Secuencia de nodos con aristas consecutivas |
| **Profundidad** (depth) | Nº de ancestros, excluido el nodo mismo; la raíz tiene depth=0 |
| **Altura** (height) | Máxima profundidad de una hoja en el subárbol |

### Árbol de ejemplo
```
        /         (raíz, depth=0)
       /    \
    home    usr        (depth=1)
    / \      |
  docs music bin      (depth=2)
  |
 CV.pdf                (hoja, depth=3)
```
- Altura del árbol = 3
- `docs` tiene 1 hijo; es nodo interno
- `CV.pdf`, `music`, `bin` son hojas

---

## 🐍 §8.2 ADT de Árbol — Clase Abstracta `Tree`

El ADT define las **operaciones mínimas** que debe proveer cualquier implementación de árbol:

| Método | Descripción |
|--------|-------------|
| `T.root()` | Posición de la raíz (o None si vacío) |
| `T.parent(p)` | Posición del padre de p (o None si p es raíz) |
| `T.children(p)` | Iterador sobre los hijos de p |
| `T.num_children(p)` | Número de hijos de p |
| `T.is_leaf(p)` | True si p no tiene hijos |
| `T.is_root(p)` | True si p es la raíz |
| `len(T)` | Número de nodos |
| `T.is_empty()` | True si el árbol está vacío |
| `T.positions()` | Iterador sobre todas las posiciones |

Los últimos cuatro se implementan **en la clase abstracta** en términos de los abstractos, mostrando el poder del diseño OOP.

In [ ]:
class Tree(metaclass=ABCMeta):
    """Clase base abstracta que representa la estructura de árbol (§8.2)."""

    # -------------------- clase Position anidada --------------------
    class Position(metaclass=ABCMeta):
        """Abstracción que representa la ubicación de un único elemento."""

        @abstractmethod
        def element(self):
            """Retorna el elemento almacenado en esta Posición."""
            raise NotImplementedError('must be implemented by subclass')

        @abstractmethod
        def __eq__(self, other):
            """True si other Position representa la misma ubicación."""
            raise NotImplementedError('must be implemented by subclass')

        def __ne__(self, other):
            return not (self == other)

    # ----- métodos abstractos que toda subclase concreta debe implementar -----
    @abstractmethod
    def root(self):
        raise NotImplementedError('must be implemented by subclass')

    @abstractmethod
    def parent(self, p):
        raise NotImplementedError('must be implemented by subclass')

    @abstractmethod
    def num_children(self, p):
        raise NotImplementedError('must be implemented by subclass')

    @abstractmethod
    def children(self, p):
        raise NotImplementedError('must be implemented by subclass')

    @abstractmethod
    def __len__(self):
        raise NotImplementedError('must be implemented by subclass')

    # ----- métodos concretos implementados en la clase base -----
    def is_root(self, p):
        return self.root() == p

    def is_leaf(self, p):
        return self.num_children(p) == 0

    def is_empty(self):
        return len(self) == 0

    def depth(self, p):
        """Profundidad de p: número de ancestros (excluido p). O(dₚ + 1)."""
        if self.is_root(p):
            return 0
        else:
            return 1 + self.depth(self.parent(p))

    def _height1(self):
        """Altura del árbol — versión INEFICIENTE O(n²)."""
        return max(self.depth(p) for p in self.positions() if self.is_leaf(p))

    def _height2(self, p):
        """Altura del subárbol en p — versión EFICIENTE O(n)."""
        if self.is_leaf(p):
            return 0
        else:
            return 1 + max(self._height2(c) for c in self.children(p))

    def height(self, p=None):
        """Altura del subárbol en p (o del árbol entero si p=None)."""
        if p is None:
            p = self.root()
        return self._height2(p)

    def positions(self):
        """Genera iteración de posiciones del árbol (breadth-first por defecto)."""
        return self.breadthfirst()

    def breadthfirst(self):
        """Genera posiciones en orden BFS (§8.4.3)."""
        if not self.is_empty():
            fringe = deque()
            fringe.append(self.root())
            while fringe:
                p = fringe.popleft()
                yield p
                for c in self.children(p):
                    fringe.append(c)

    def __iter__(self):
        for p in self.positions():
            yield p.element()

print("Clase abstracta Tree definida ✓")
print(f"Métodos abstractos: root, parent, num_children, children, __len__")
print(f"Métodos concretos:  is_root, is_leaf, is_empty, depth, height, positions, breadthfirst")

---

## 📐 §8.2.1–8.2.2 Profundidad y Altura

### `depth(p)` — O(dₚ + 1)

```python
def depth(self, p):
    if self.is_root(p):    # caso base
        return 0
    else:
        return 1 + self.depth(self.parent(p))   # recursión
```

La llamada asciende hasta la raíz → **T(p) = O(dₚ + 1)**

### `height1` vs `height2`

| Versión | Idea | Complejidad |
|---------|------|-------------|
| `_height1` | Para cada hoja, calcula `depth(p)` — O(n) hojas × O(n) cada una | **O(n²)** |
| `_height2` | Recursión bottom-up: height(p) = 1 + max(height(c)) | **O(n)** |

**¿Por qué height2 es O(n)?**  
Cada nodo p se visita exactamente una vez. Total operaciones = n. (Prop 8.5: Σ num_children = n−1)

In [ ]:
# ── Implementación mínima de árbol general para demostraciones ──────────────

class SimpleTree(Tree):
    """Árbol general simple para pruebas (raíz + hijos como lista)."""

    class _Node:
        __slots__ = '_element', '_parent', '_children'
        def __init__(self, element, parent=None):
            self._element = element
            self._parent  = parent
            self._children = []

    class _Position(Tree.Position):
        def __init__(self, container, node):
            self._container = container
            self._node = node
        def element(self):
            return self._node._element
        def __eq__(self, other):
            return type(other) is type(self) and other._node is self._node

    def _validate(self, p):
        if not isinstance(p, self._Position):
            raise TypeError('p must be proper Position type')
        if p._container is not self:
            raise ValueError('p does not belong to this container')
        return p._node

    def _make_position(self, node):
        return self._Position(self, node) if node is not None else None

    def __init__(self):
        self._root = None
        self._size = 0

    def __len__(self):
        return self._size

    def root(self):
        return self._make_position(self._root)

    def parent(self, p):
        node = self._validate(p)
        return self._make_position(node._parent)

    def num_children(self, p):
        node = self._validate(p)
        return len(node._children)

    def children(self, p):
        node = self._validate(p)
        for child in node._children:
            yield self._make_position(child)

    def add_root(self, e):
        if self._root is not None:
            raise ValueError('Root exists')
        self._size = 1
        self._root = self._Node(e)
        return self._make_position(self._root)

    def add_child(self, p, e):
        node = self._validate(p)
        self._size += 1
        child = self._Node(e, parent=node)
        node._children.append(child)
        return self._make_position(child)


# ── Construir árbol del ejemplo del libro (sistema de archivos) ──────────────
T = SimpleTree()
root  = T.add_root("/")
home  = T.add_child(root, "home")
usr   = T.add_child(root, "usr")
docs  = T.add_child(home, "docs")
music = T.add_child(home, "music")
binn  = T.add_child(usr,  "bin")
cv    = T.add_child(docs, "CV.pdf")
song  = T.add_child(music, "song.mp3")

print(f"Árbol construido: {len(T)} nodos")
print()

# Verificar depth y height
for label, pos in [("/", root), ("home", home), ("docs", docs),
                   ("CV.pdf", cv), ("usr", usr), ("bin", binn)]:
    d = T.depth(pos)
    h = T.height(pos)
    leaf = "🍃 hoja" if T.is_leaf(pos) else "🔵 interno"
    print(f"  {label:10s}  depth={d}  height={h}  {leaf}")

print(f"\nAltura del árbol:       {T.height()}")
print(f"Altura (method _height1): {T._height1()}")

---

## 📊 Prop. 8.5 — Verificación numérica

> **Proposición 8.5:** En un árbol T con n nodos, la suma del número de hijos de todos los nodos es **n − 1**.

*Consecuencia:* En height2, la línea `max(self._height2(c) for c in self.children(p))` se ejecuta exactamente n-1 veces en total → **O(n)**.

In [ ]:
# Verificación de la Proposición 8.5 para varios tamaños de árbol
import random

def build_random_tree(n, max_children=4):
    """Construye árbol aleatorio de n nodos."""
    t = SimpleTree()
    root = t.add_root(0)
    nodes = [root]
    for i in range(1, n):
        parent = random.choice(nodes)
        child = t.add_child(parent, i)
        nodes.append(child)
    return t

print("Verificación de la Proposición 8.5: Σ num_children(p) = n − 1")
print(f"{'n':>6} | {'Σ hijos':>10} | {'n-1':>10} | {'OK':>6}")
print("-" * 40)

random.seed(42)
for n in [5, 10, 20, 50, 100]:
    t = build_random_tree(n)
    total_children = sum(t.num_children(p) for p in t.positions())
    ok = "✓" if total_children == n - 1 else "✗"
    print(f"{n:>6} | {total_children:>10} | {n-1:>10} | {ok:>6}")

In [ ]:
# Comparación empírica: height1 O(n²) vs height2 O(n)
import time

ns = [100, 200, 400, 800, 1600]
times_h1, times_h2 = [], []

random.seed(0)
for n in ns:
    t = build_random_tree(n, max_children=3)

    t0 = time.perf_counter()
    for _ in range(20): t._height1()
    times_h1.append((time.perf_counter() - t0) / 20 * 1e6)

    t0 = time.perf_counter()
    for _ in range(20): t.height()
    times_h2.append((time.perf_counter() - t0) / 20 * 1e6)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ns, times_h1, 'r-o', label='height1 O(n²)')
axes[0].plot(ns, times_h2, 'b-s', label='height2 O(n)')
axes[0].set_xlabel('n (nodos)')
axes[0].set_ylabel('Tiempo (μs)')
axes[0].set_title('height1 vs height2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# log-log para ver pendiente
axes[1].loglog(ns, times_h1, 'r-o', label='height1')
axes[1].loglog(ns, times_h2, 'b-s', label='height2')
slope1 = np.polyfit(np.log(ns), np.log(times_h1), 1)[0]
slope2 = np.polyfit(np.log(ns), np.log(times_h2), 1)[0]
axes[1].set_xlabel('n (log)')
axes[1].set_ylabel('Tiempo (log)')
axes[1].set_title(f'Escala log-log\npendiente height1≈{slope1:.2f}, height2≈{slope2:.2f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Pendientes log-log: height1 ≈ {slope1:.2f} (~2), height2 ≈ {slope2:.2f} (~1)")

---

## 🧪 Tests

In [ ]:
import unittest

class TestTreeADT(unittest.TestCase):

    def setUp(self):
        """Árbol de prueba:
              A           depth=0, internal
             / \
            B   C         depth=1, B=internal, C=leaf
           /|
          D  E            depth=2, D=leaf, E=leaf
        """
        self.T = SimpleTree()
        A = self.T.add_root('A')
        B = self.T.add_child(A, 'B')
        C = self.T.add_child(A, 'C')
        D = self.T.add_child(B, 'D')
        E = self.T.add_child(B, 'E')
        self.A, self.B, self.C, self.D, self.E = A, B, C, D, E

    def test_size(self):
        self.assertEqual(len(self.T), 5)

    def test_root(self):
        self.assertEqual(self.T.root().element(), 'A')
        self.assertTrue(self.T.is_root(self.A))
        self.assertFalse(self.T.is_root(self.B))

    def test_depth(self):
        self.assertEqual(self.T.depth(self.A), 0)
        self.assertEqual(self.T.depth(self.B), 1)
        self.assertEqual(self.T.depth(self.C), 1)
        self.assertEqual(self.T.depth(self.D), 2)

    def test_height(self):
        self.assertEqual(self.T.height(), 2)
        self.assertEqual(self.T.height(self.B), 1)
        self.assertEqual(self.T.height(self.C), 0)   # hoja

    def test_height1_equals_height2(self):
        self.assertEqual(self.T._height1(), self.T.height())

    def test_is_leaf(self):
        self.assertFalse(self.T.is_leaf(self.A))
        self.assertFalse(self.T.is_leaf(self.B))
        self.assertTrue(self.T.is_leaf(self.C))
        self.assertTrue(self.T.is_leaf(self.D))
        self.assertTrue(self.T.is_leaf(self.E))

    def test_num_children(self):
        self.assertEqual(self.T.num_children(self.A), 2)
        self.assertEqual(self.T.num_children(self.B), 2)
        self.assertEqual(self.T.num_children(self.C), 0)

    def test_prop_8_5(self):
        """Prop 8.5: suma de hijos = n-1."""
        total = sum(self.T.num_children(p) for p in self.T.positions())
        self.assertEqual(total, len(self.T) - 1)

    def test_iter(self):
        elements = set(self.T)
        self.assertEqual(elements, {'A', 'B', 'C', 'D', 'E'})

suite = unittest.TestLoader().loadTestsFromTestCase(TestTreeADT)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'✅ Todos los tests pasaron' if result.wasSuccessful() else '❌ Hubo fallos'}")

---

## 📋 Resumen

| Operación | Complejidad | Notas |
|-----------|-------------|-------|
| `is_root`, `is_leaf`, `is_empty` | O(1) | Implementadas en la ABC |
| `root`, `parent`, `num_children`, `children` | O(1)* | *Depende de la implementación concreta |
| `depth(p)` | O(dₚ + 1) | Recursión ascendente |
| `_height1()` | O(n²) | Llama depth en cada hoja |
| `height()` / `_height2(p)` | O(n) | Bottom-up, visita cada nodo una vez |
| `breadthfirst()` | O(n) | Cola FIFO |

**Diseño OOP clave:**  
La clase `Tree` es una *Abstract Base Class* (ABC). Sus subclases concretas solo implementan los 5 métodos abstractos; todos los demás se heredan gratis. Esto es el **patrón Template Method** en acción.

## 📚 Referencias

- Goodrich, M., Tamassia, R., & Goldwasser, M. (2013). *Data Structures and Algorithms in Python*. Wiley. §8.1–§8.2
- Documentación Python ABCs: https://docs.python.org/3/library/abc.html

## ✅ Autoevaluación

Antes de continuar al notebook siguiente, comprobá que podés:

- [ ] Dibujar un árbol de 8 nodos e indicar raíz, hojas, internos, depth y height de cada nodo
- [ ] Calcular a mano `depth(p)` y `height(p)` para cualquier nodo
- [ ] Explicar la diferencia entre `height1` O(n²) y `height2` O(n) usando Prop 8.5
- [ ] Explicar qué son los métodos abstractos y por qué se usan en `Tree`
- [ ] Enunciar la Prop. 8.5 y usarla para justificar O(n) de `height2`

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>